# Exploración de la API de ESIOS

Objetivo:

- Comprender la estructura del JSON.
- Identificar los campos relevantes.
- Detectar problemas de calidad.
- Definir las transformaciones necesarias para el pipeline.

In [ ]:
import requests
import json
import pandas as pd
from pprint import pprint
from dotenv import load_dotenv
import os

In [ ]:
load_dotenv()
API_KEY = os.getenv("ESIOS_API_KEY")

In [ ]:
#LLamada simple a la API de ESIOS para obtener una lista de indicadores
HEADERS = {
    "Accept": "application/json; application/vnd.esios-api-v1+json",
    "Content-Type": "application/json",
    "x-api-key": API_KEY
}
url = "https://api.esios.ree.es/indicators"
response = requests.get(
    url,
    headers=HEADERS,
)
response_json = response.json()
pprint(response_json)

In [ ]:
# LLamada a la API de ESIOS para obtener los datos de un indicador específico
indicator_id = 1293
url = f"https://api.esios.ree.es/indicators/{indicator_id}"
PARAMS = {
    "start_date": "2026-01-01T00:00:00",
    "end_date": "2026-01-31T23:59:59",
    "geo_ids": "ES",
    "time_trunc": "hour"
}
response = requests.get(
    url,
    headers=HEADERS,
    params=PARAMS
)
response_json = response.json()
pprint(response_json)

In [ ]:
# Guardar la respuesta en un archivo JSON data/raw/sample_response.json
with open("../data/raw/sample_response.json", "w") as f:
    json.dump(response_json, f, indent=4)


In [ ]:
# Convertir los datos de serie temporal a un DataFrame de pandas
df = pd.DataFrame(response_json['indicator']['values']) 
print(df.head()) # Mostrar las primeras filas del DataFrame
print(df.info()) # Mostrar información del DataFrame
print(df.describe()) # Mostrar estadísticas descriptivas del DataFrame 

In [ ]:
df.describe()

| Columna  | Tipo     | ¿Se conservará? | Motivo                 |
| -------- | -------- | --------------- | ---------------------- |
| datetime | str | No             | Evitar problemas con cambios de horario        |
| value    | float   | Sí              | Métrica                |
| geo_id   | int      | Sí              | Dimensión geográfica   |
| geo_name     | str     | Si            | Se conserva pero sera normalizado posteriormente |
| datetime_utc    | str     | Si            | Clave temporal universal |
| tz_time    | str      | No           | innecesario ya que se utilizara UTC |


## Analisis de calidad de los datos

In [ ]:
df.isnull().sum() # Contar valores nulos en cada columna

In [ ]:
df.duplicated().sum() # Contar valores duplicados en el DataFrame

In [ ]:
df.dtypes # Mostrar los tipos de datos de cada columna

## Conclusiones

- La informacion util del indicador se encuentra en la coleccion values
- Sera necesario eliminar las columnas innecesarias
- sera necesario convertir fechas a formatos estandar
- No se detectaron valores nulos ni registros duplicados. No obstante, el pipeline incorporará validaciones de calidad para garantizar la integridad de los datos ante posibles cambios en la API o incidencias futuras